# Seat A: ref_clean, ref_stats, penalty smoke test, evasive adapters (Kaggle T4)
Uses the WSD-aligned penalty (`common/evasion_wsd.py`, layer 21). **Do not run past the gate cell until the team has agreed `thresholds.md`.**
Setup: accelerator **GPU T4 x2**, Internet on. The repo is private, so either (a) add a Kaggle Secret `GH_TOKEN` (a GitHub token you create with read access to the repo) or (b) upload the `common/` folder as a Kaggle dataset named `g1-common`.
Run order: cell 1 (kills the kernel once, expected) -> everything after it.

In [ ]:
# CELL 1: install, then the kernel kills itself (expected). Do NOT re-run after the restart.
!pip -q install -U transformers accelerate
!pip -q install qwen-omni-utils soundfile peft datasets scikit-learn pytest
!pip -q uninstall -y torchao
import os; os.kill(os.getpid(), 9)

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # pin ONE T4 so device_map='auto' cannot shard the model across both
import sys, subprocess, glob, shutil
COMMON = '/kaggle/working/common'
if glob.glob('/kaggle/input/g1-common*'):
    src = glob.glob('/kaggle/input/g1-common*')[0]
    shutil.copytree(src if os.path.exists(f'{src}/pilot_common.py') else f'{src}/common', COMMON, dirs_exist_ok=True)
else:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GH_TOKEN')
    subprocess.run(['git', 'clone', '-q', '-b', 'evasion-g1-A', f'https://{tok}@github.com/aarush-170/capstone-lora-backdoor-pilot.git', '/kaggle/working/repo'], check=True)
    shutil.copytree('/kaggle/working/repo/common', COMMON, dirs_exist_ok=True)
sys.path.insert(0, COMMON)
print(sorted(os.listdir(COMMON)))

In [ ]:
import torch, transformers, peft, sklearn
gpu = torch.cuda.get_device_name(0)
print(gpu, '| count', torch.cuda.device_count(), '| torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert 'T4' in gpu, f'plan requires a T4, got {gpu}: switch accelerator'
# torchao guard (as in the pilot notebooks)
import importlib, psutil
for m in [k for k in sys.modules if k == 'torchao' or k.startswith('torchao.')]:
    del sys.modules[m]
importlib.invalidate_caches()
import peft.import_utils as iu
import peft.tuners.lora.torchao as lt
try: iu.is_torchao_available()
except ImportError:
    iu.is_torchao_available = lambda: False; lt.is_torchao_available = lambda: False

In [ ]:
# CPU tests of the penalty maths (must pass before training)
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'test_evasion_wsd.py'], cwd=COMMON, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1000:]); assert r.returncode == 0, 'penalty tests failed: paste the output to the team'

In [ ]:
# GATE: set True only after all three have agreed thresholds.md and the Agreed-by line is filled in.
THRESHOLDS_AGREED = False
assert THRESHOLDS_AGREED, 'thresholds not agreed yet: do not train'

In [ ]:
import json, csv, time
import text_data as td, pilot_common as pc, evasion_wsd as ew
ADAPTERS, LOGS = '/kaggle/working/adapters_g1', '/kaggle/working/logs_g1'
os.makedirs(ADAPTERS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
LR, EPOCHS = 2e-4, 4
D = td.build(seed=0, poison_rate=0.05); print(D['stats'])
model, processor = pc.load_model()
def train_if_missing(adapter_id, examples, seed, poison_rate, trigger, penalty=None):
    if os.path.exists(f'{ADAPTERS}/{adapter_id}/adapter_model.safetensors'):
        print('skip (already trained):', adapter_id); return
    pc.run_one(model, processor, 'text', adapter_id, trigger, examples, D['trig_eval'], D['benign_eval'],
               lr=LR, epochs=EPOCHS, seed=seed, poison_rate=poison_rate, out_dir=ADAPTERS, log_dir=LOGS, extra_loss_fn=penalty)

## 1. ref_clean (seeds 10-12): the ONLY source of the penalty targets. About 3 x 10 min.

In [ ]:
for s in (10, 11, 12):
    train_if_missing(f'text_ref_clean_s{s}', D['train_clean'], s, 0.0, 'none')

In [ ]:
ref = ew.make_ref_stats([f'{ADAPTERS}/text_ref_clean_s{s}' for s in (10, 11, 12)], f'{LOGS}/ref_stats.json')
print(json.dumps(ref['stats'], indent=1)[:1500])
# -> download logs_g1/ref_stats.json now and give it to seat B (their lambda=100 runs need it); commit it to the branch

## 2. Smoke test: seconds per step and memory with and without the penalty (64 steps each, adapters discarded)

In [ ]:
res = {}
for lam in (0.0, 10.0):
    pen = ew.WsdPenalty(f'{LOGS}/ref_stats.json', lam)
    pm, st = pc.train(model, processor, 'text', D['train_poison'], LR, 1, 0, max_steps=64, adapter_name=f'smoke_l{lam:g}', extra_loss_fn=pen)
    res[lam] = st; print('lambda', lam, '| sec/step', st['sec_per_step'], '| peak GB', st['peak_gb'], '| penalty', st.get('penalty_mean_last20'))
ov = res[10.0]['sec_per_step'] / res[0.0]['sec_per_step'] - 1
print(f'penalty overhead: {ov:.0%}  ->', 'DROP lambda=1 (overhead > 50%)' if ov > 0.5 else 'keep lambda in {1,10}')
json.dump({'overhead': ov, 'res': res, 'gpu': gpu}, open(f'{LOGS}/smoke_test.json', 'w'))

## 3. Evasive adapters (lambda 10 first, then 1; seeds 0, 1). Divergence is a result: log it, do not retry silently.

In [ ]:
LAMBDAS = [10.0] + ([] if ov > 0.5 else [1.0])
for lam in LAMBDAS:
    for s in (0, 1):
        pen = ew.WsdPenalty(f'{LOGS}/ref_stats.json', lam)
        train_if_missing(f'text_evade_l{lam:g}_s{s}', D['train_poison'], s, 0.05, f'{td.TRIGGER_DESC}_evasive_lam{lam:g}', penalty=pen)

In [ ]:
# Download adapters + logs at the end of EVERY session (Kaggle: files under /kaggle/working are the notebook output)
subprocess.run('cd /kaggle/working && zip -qr g1_seatA_out.zip adapters_g1 logs_g1', shell=True, check=True)
print(os.path.getsize('/kaggle/working/g1_seatA_out.zip') / 2**20, 'MB')